# 🚨 Opération Redressement — notebook d'agence
**Module Predictive Analytics & Data Intelligence · M2 · Pairagogie du 13/11/2026**

Votre agence est missionnée par une marque **en déclin**. Ce notebook est **pré-codé** : vous exécutez les cellules (▶), vous lisez les résultats, et vous **prenez les décisions** dans les cellules marquées **✋ À VOUS**.

| Étape | Question | Livrable |
|---|---|---|
| 1 | Les données sont-elles fiables ? | rapport de nettoyage |
| 2 | D'où vient le déclin ? Structurel ou conjoncturel ? | diagnostic + prévision |
| 3 | Qui va partir ? (construire la cible) | taux de départ |
| 4 | Quels signaux annoncent le départ ? | 3 graphiques |
| 5 | Quelles variables a-t-on le droit d'utiliser ? | liste justifiée |
| 6 | Quel modèle est le plus fiable ? | tableau des 3 modèles |
| 7 | Quelle offre pour quel segment ? | next best offer |
| 8 | Qui cibler avec le budget ? | plan chiffré |

> ⚠️ Toutes les données sont **fictives**, créées pour l'exercice.

## 0 · Choisir sa marque et charger les fichiers

In [1]:
# ✋ À VOUS : écrivez le code de votre marque entre guillemets
#    "velin"  "atelier_nord"  "krousty"  "flixa"  "spark"
MARQUE = "atelier_nord"

In [2]:
import os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_score, recall_score
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
COULEURS = ["#1f4e79", "#e07b39", "#3a9e6f", "#8064a2", "#c0504d"]

CONFIG = {
  # budget de la campagne (€), coût d'envoi par contact (€), valeur d'un client retenu (marge, €) — hypothèses du client
  "velin":        dict(nom="Maison Vélin", fenetre=180, date="date",    montant="montant_eur", budget=8000,   cout_contact=2.0, valeur=240),
  "atelier_nord": dict(nom="Atelier Nord", fenetre=120, date="date",    montant="montant_eur", budget=3500,   cout_contact=0.5, valeur=140),
  "krousty":      dict(nom="Krousty",      fenetre=60,  date="date",    montant="montant_eur", budget=1500,   cout_contact=0.3, valeur=70),
  "flixa":        dict(nom="Flixa",        fenetre=90,  date="date",    montant="montant_eur", budget=1200,   cout_contact=0.5, valeur=60),
  "spark":        dict(nom="Spark",        fenetre=60,  date="semaine", montant="achat_eur",   budget=800,    cout_contact=0.2, valeur=40),
}
C = CONFIG[MARQUE]
REF = pd.Timestamp("2026-03-31")      # date d'observation : on « se place » au 31 mars 2026
FIN_FENETRE = REF + pd.Timedelta(days=C["fenetre"])

# --- Chargement : sur Colab, une fenêtre s'ouvre pour déposer vos 5 fichiers CSV
DOSSIER = os.environ.get("DATA_DIR", ".")
try:
    from google.colab import files
    if not os.path.exists(f"{MARQUE}_clients.csv"):
        files.upload()
except ImportError:
    pass
lire = lambda f: pd.read_csv(os.path.join(DOSSIER, f"{MARQUE}_{f}.csv"))
clients, activite, digital = lire("clients"), lire("activite"), lire("digital")
campagne, mensuel = lire("campagne_test"), lire("mensuel")
for nom, t in [("clients", clients), ("activite", activite), ("digital", digital), ("campagne_test", campagne), ("mensuel", mensuel)]:
    print(f"{nom:<14} {t.shape[0]:>7} lignes × {t.shape[1]} colonnes")
print(f"\nMarque : {C['nom']} · fenêtre de départ : {C['fenetre']} jours après le {REF.date()}")

FileNotFoundError: [Errno 2] No such file or directory: '.\\atelier_nord_clients.csv'

In [ ]:
clients.head()

## 1 · Nettoyer : peut-on faire confiance aux données ?
Un data analyst passe souvent **plus de la moitié de son temps** à nettoyer. Exécutez, puis notez les chiffres dans votre note de cadrage.

In [ ]:
avant = len(clients)
clients = clients.drop_duplicates()
doublons = avant - len(clients)
# dates dans deux formats (2023-05-12 et 12/05/2023)
iso = pd.to_datetime(clients["date_inscription"], format="%Y-%m-%d", errors="coerce")
fr  = pd.to_datetime(clients["date_inscription"], format="%d/%m/%Y", errors="coerce")
clients["date_inscription"] = iso.fillna(fr)
# âges impossibles
clients["age"] = pd.to_numeric(clients["age"], errors="coerce")
ages_faux = ((clients["age"] < 15) | (clients["age"] > 100)).sum()
clients.loc[(clients["age"] < 15) | (clients["age"] > 100), "age"] = np.nan
clients["age"] = clients["age"].fillna(clients["age"].median())
# villes mal saisies
villes_avant = clients["ville"].nunique()
clients["ville"] = clients["ville"].str.strip().str.title()
print(f"Doublons supprimés        : {doublons}")
print(f"Âges impossibles corrigés : {ages_faux} (remplacés par la médiane)")
print(f"Villes : {villes_avant} orthographes → {clients['ville'].nunique()} villes")
print(f"Clients sans fiche digitale : {len(set(clients.client_id) - set(digital.client_id))}")
print("\nValeurs manquantes restantes (%) :")
print((clients.isna().mean() * 100).round(1)[lambda s: s > 0])

## 2 · Diagnostiquer : d'où vient le déclin ?
Le fichier mensuel retrace toute la clientèle de 2021 à 2026. **Une question = un graphique.**
- Quel segment décroche ? Depuis quand ?
- La baisse est-elle **structurelle** (tendance durable) ou **conjoncturelle** (accident ponctuel) ?

In [ ]:
segs = [c for c in mensuel.columns if c != "mois" and not c.startswith(("ca_total", "ca_keur", "marge", "depenses", "nouveaux", "part_", "frequentation", "telechargements"))][:3]
mensuel["date"] = pd.to_datetime(mensuel["mois"])
fig, ax = plt.subplots()
for i, s in enumerate(segs):
    ax.plot(mensuel["date"], mensuel[s], label=s, color=COULEURS[i], lw=2)
ax.set_title(f"{C['nom']} — évolution mensuelle par segment (données fictives)")
ax.legend(frameon=False); plt.show()

mensuel["annee"] = mensuel["date"].dt.year
an = mensuel[mensuel.annee.between(2022, 2025)].groupby("annee")[segs].sum()
print("Total annuel par segment :")
print(an.round(0))
evo = pd.DataFrame({"2023 → 2025 (%)": (an.loc[2025] / an.loc[2023] - 1) * 100,
                    "2024 → 2025 (%)": (an.loc[2025] / an.loc[2024] - 1) * 100}).replace([np.inf, -np.inf], np.nan).round(1)
print("\nÉvolution (vide = segment lancé récemment) :")
print(evo)

In [ ]:
# Prévision « si rien ne change » : tendance des 24 derniers mois, corrigée de la saisonnalité
from sklearn.linear_model import LinearRegression
total = mensuel[segs].sum(axis=1)
saison = total.groupby(mensuel["date"].dt.month).mean() / total.mean()
desais = total / mensuel["date"].dt.month.map(saison)
recent = mensuel.index[-24:]
lr = LinearRegression().fit(np.array(recent).reshape(-1, 1), desais[recent])
futur = np.arange(len(mensuel), len(mensuel) + 12)
mois_futurs = pd.date_range(mensuel["date"].iloc[-1] + pd.offsets.MonthBegin(1), periods=12, freq="MS")
prev = lr.predict(futur.reshape(-1, 1)) * mois_futurs.month.map(saison).values
fig, ax = plt.subplots()
ax.plot(mensuel["date"], total, color=COULEURS[0], lw=2, label="réel")
ax.plot(mois_futurs, prev, color=COULEURS[1], lw=2, ls="--", label="prévision 12 mois")
ax.set_title(f"{C['nom']} — total et prévision si rien ne change"); ax.legend(frameon=False); plt.show()
derniers12 = total.iloc[-12:].sum()
print(f"12 derniers mois : {derniers12:,.0f} · 12 prochains mois prévus : {prev.sum():,.0f} → {(prev.sum()/derniers12-1)*100:+.1f} %")

✋ **À VOUS (note de cadrage)** : en 3 phrases — quel segment fait chuter la marque, depuis quand, et le creux visible sur le graphique est-il structurel ou conjoncturel ?

## 3 · Construire la cible : qui est « parti » ?
Il n'y a **pas de colonne toute faite**. On se place au **31 mars 2026** :
- **Population** : les clients **actifs** (au moins une activité dans les 12 mois avant le 31/03/2026). Les autres sont déjà partis : on ne les prédit pas.
- **Cible Y = 1** : aucune activité dans la **fenêtre** qui suit (voir la durée de votre marque).

In [ ]:
act = activite.copy()
act["d"] = pd.to_datetime(act[C["date"]])
if "statut_paiement" in act:                       # Flixa : seuls les paiements réussis comptent
    act_ok = act[act["statut_paiement"] == "ok"]
else:
    act_ok = act
avant12 = act_ok[(act_ok.d > REF - pd.Timedelta(days=365)) & (act_ok.d <= REF)]
dans_fenetre = act_ok[(act_ok.d > REF) & (act_ok.d <= FIN_FENETRE)]
actifs = avant12.client_id.unique()
base = pd.DataFrame({"client_id": actifs})
base["Y"] = (~base.client_id.isin(dans_fenetre.client_id)).astype(int)
print(f"Clients dans le fichier : {clients.client_id.nunique()} · actifs au 31/03/2026 : {len(base)}")
print(f"Taux de départ (Y = 1) dans les {C['fenetre']} jours : {base.Y.mean():.1%}")

### Variables de comportement — uniquement **avant** le 31/03/2026
Utiliser une donnée d'après le 31 mars, ce serait **lire la copie du voisin** : c'est une **fuite de données**.

In [ ]:
h = act[(act.d <= REF) & (act.d > REF - pd.Timedelta(days=365))]
g = h.groupby("client_id")
feat = pd.DataFrame({
    "recence_jours": (REF - g.d.max()).dt.days,
    "frequence_12m": g.size(),
    "montant_12m": g[C["montant"]].sum(),
    "activite_3_derniers_mois": h[h.d > REF - pd.Timedelta(days=91)].groupby("client_id").size(),
})
feat["activite_3_derniers_mois"] = feat["activite_3_derniers_mois"].fillna(0)
feat["tendance"] = (feat["activite_3_derniers_mois"] / (feat["frequence_12m"] / 4)).round(2)   # 1 = stable, < 1 = ralentit
# variables numériques propres à votre marque (moyenne sur 12 mois)
for c in h.select_dtypes("number").columns:
    if c not in [C["montant"]]:
        feat[f"moy_{c}"] = g[c].mean()
if "statut_paiement" in h:
    feat["echecs_paiement_12m"] = h[h.statut_paiement == "echec"].groupby("client_id").size()
    feat["echecs_paiement_12m"] = feat["echecs_paiement_12m"].fillna(0)
if C["montant"] in h and (h[C["montant"]] < 0).any():
    feat["part_retours"] = h.assign(r=h[C["montant"]] < 0).groupby("client_id").r.mean()

data = (base.merge(feat.reset_index(), on="client_id", how="left")
            .merge(clients.drop_duplicates("client_id"), on="client_id", how="left")
            .merge(digital, on="client_id", how="left"))
data["anciennete_mois"] = ((REF - data["date_inscription"]).dt.days / 30.4).round(0)
data = data.drop(columns=["date_inscription"])
print(data.shape); data.head()

## 4 · Explorer : quels signaux annoncent le départ ?
Choisissez les **3 graphiques** qui racontent le mieux votre histoire. Titre = le message, pas la variable.

In [ ]:
def taux_par(var, n_classes=4):
    s = data[var]
    if s.dtype.kind in "if" and s.nunique() > 8:
        s = pd.qcut(s.rank(method="first"), n_classes, labels=[f"Q{i+1}" for i in range(n_classes)])
    t = data.groupby(s.astype(str))["Y"].agg(["mean", "size"]).rename(columns={"mean": "taux_depart", "size": "nb"})
    return t

# ✋ À VOUS : changez la variable explorée (copiez un nom de colonne de data.head())
VARIABLE = "tendance"
t = taux_par(VARIABLE)
ax = (t["taux_depart"] * 100).plot.bar(color=COULEURS[0], rot=0)
ax.axhline(data.Y.mean() * 100, color=COULEURS[1], ls="--", label="moyenne")
ax.set_ylabel("% de départs"); ax.set_title(f"Taux de départ selon {VARIABLE}"); ax.legend(frameon=False); plt.show()
t

In [ ]:
# Classement automatique : quelles variables séparent le plus les partants des fidèles ?
ecarts = {}
for v in data.columns.drop(["client_id", "Y"]):
    try:
        t = taux_par(v); t = t[t.nb >= 50]
        ecarts[v] = round((t.taux_depart.max() - t.taux_depart.min()) * 100, 1)
    except Exception:
        pass
pd.Series(ecarts, name="écart max (points)").sort_values(ascending=False).head(15)

## 5 · Choisir les variables : a-t-on le **droit** de tout utiliser ?
Toutes les colonnes de vos fichiers n'ont pas leur place dans un modèle. Certaines posent un problème de **logique**, d'autres un problème **juridique ou éthique**, une autre n'a tout simplement **rien à faire là**. Combien sont-elles ? Lesquelles ? Pourquoi ? C'est à vous de le démontrer.

✋ **À VOUS** : écrivez les colonnes à écarter dans la liste ci-dessous, puis relancez les étapes 6 à 8. Comparez les résultats **avant / après** et justifiez chaque exclusion.

In [ ]:
VARIABLES = [c for c in data.columns if c not in ["client_id", "Y"]]
# ✋ À VOUS : écrivez ici les variables à exclure, par ex. EXCLURE = ["nom_de_colonne", ...]
EXCLURE = []
VARIABLES = [v for v in VARIABLES if v not in EXCLURE]
print(len(VARIABLES), "variables utilisées :", VARIABLES)

## 6 · Entraîner et comparer 3 modèles
- **Train / test** : on apprend sur 70 % des clients, on vérifie sur 30 % jamais vus (les annales vs l'examen).
- **Validation croisée** : on répète 5 fois sur des découpages différents → le score est-il **stable** ?
- **AUC** : 0,5 = pile ou face · 0,7-0,8 = bon · > 0,93 = **trop beau pour être vrai**

In [ ]:
X = pd.get_dummies(data[VARIABLES], drop_first=True, dtype=float)
X = X.fillna(X.median())
y = data["Y"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

MODELES = {
    "Régression logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Arbre de décision":     DecisionTreeClassifier(max_depth=4, min_samples_leaf=40, random_state=42),
    "Forêt aléatoire":       RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1),
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
res, probas = [], {}
for nom, m in MODELES.items():
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]; probas[nom] = p
    auc_cv = cross_val_score(m, X_tr, y_tr, cv=cv, scoring="roc_auc")
    res.append({"modèle": nom, "AUC train": roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1]),
                "AUC test": roc_auc_score(y_te, p), "AUC validation croisée": auc_cv.mean(), "écart-type CV": auc_cv.std()})
res = pd.DataFrame(res).set_index("modèle").round(3)
if (res["AUC test"] > 0.93).any():
    print("🚨 AUC > 0,93 : résultat suspect. Avant de vous réjouir, demandez-vous pourquoi. Retournez à l'étape 5.")
res

In [ ]:
# ✋ À VOUS : choisissez le modèle retenu (simple, stable, explicable) et le seuil
MODELE_RETENU = "Régression logistique"
SEUIL = 0.5
p = probas[MODELE_RETENU]; pred = (p >= SEUIL).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
print(f"Matrice de confusion (test, seuil {SEUIL}) :")
print(f"  partants bien repérés (VP) : {tp:>4}   partants ratés (FN)   : {fn:>4}")
print(f"  fausses alertes (FP)       : {fp:>4}   fidèles bien vus (VN) : {tn:>4}")
print(f"Exactitude {((tp+tn)/len(y_te)):.1%} · modèle naïf « personne ne part » {1-y_te.mean():.1%}")
print(f"Rappel {recall_score(y_te, pred):.1%} · Précision {precision_score(y_te, pred, zero_division=0):.1%}")

In [ ]:
# Qu'est-ce qui pèse dans le modèle ? (forêt aléatoire)
imp = pd.Series(MODELES["Forêt aléatoire"].feature_importances_, index=X.columns).sort_values()
imp.tail(10).plot.barh(color=COULEURS[0]); plt.title("Les 10 signaux les plus utiles au modèle"); plt.show()

## 7 · Quelle offre pour quel segment ? (campagne test)
Le client a testé 3 offres + un groupe **contrôle** (aucune offre). L'**effet réel** d'une offre = taux de réaction de l'offre **moins** taux du contrôle (*uplift*).

In [ ]:
tx = campagne.pivot_table(index="segment", columns="offre", values="a_reagi_30j", aggfunc="mean")
uplift = tx.drop(columns="controle").sub(tx["controle"], axis=0)
cout_offre = campagne.groupby("offre").cout_offre_eur.mean()
# gain net par client contacté = clients retenus en plus × valeur d'un client retenu − coût de l'offre
gain_contact = uplift * C["valeur"] - cout_offre[uplift.columns]
print("Taux de réaction (%) :"); print((tx * 100).round(1))
print("\nUplift vs contrôle (points) :"); print((uplift * 100).round(1))
print("\nCoût unitaire des offres (€) :"); print(cout_offre.drop("controle").round(2))
print(f"\nGain net par contact (€) = uplift × {C['valeur']} € − coût de l'offre :"); print(gain_contact.round(2))
nbo = gain_contact.idxmax(axis=1)
print("\n➡ Next best offer par segment :"); print(nbo)
ax = (uplift * 100).plot.bar(rot=0, color=COULEURS[:3]); ax.set_ylabel("points de % en plus du contrôle")
ax.set_title("Effet réel des offres par segment (uplift)"); ax.legend(frameon=False); plt.show()

## 8 · Qui cibler avec le budget ?
On classe les clients du test du plus risqué au moins risqué, puis on extrapole à toute la base active.
Hypothèses du client : **budget**, **coût d'envoi par contact**, **valeur d'un client retenu**. On y ajoute le coût de l'**offre retenue** ; le taux de succès = son uplift moyen.

In [ ]:
# ✋ À VOUS : l'offre que vous recommandez (par défaut : celle qui rapporte le plus en moyenne)
OFFRE_RETENUE = gain_contact.mean().idxmax()
TAUX_SUCCES = float(uplift[OFFRE_RETENUE].mean())        # uplift moyen de l'offre, en proportion
BUDGET, VALEUR = C["budget"], C["valeur"]
COUT = C["cout_contact"] + float(cout_offre[OFFRE_RETENUE])
print(f"Offre retenue : {OFFRE_RETENUE} · coût total par client ciblé : {COUT:.2f} €")
facteur = len(data) / len(y_te)
ordre = np.argsort(-probas[MODELE_RETENU]); y_ord = y_te.values[ordre]
lignes = []
for pct in range(5, 101, 5):
    k = int(len(y_ord) * pct / 100)
    n_cibles = k * facteur; partants = y_ord[:k].sum() * facteur
    cout_total = n_cibles * COUT; sauves = partants * TAUX_SUCCES
    lignes.append({"% ciblé": pct, "clients ciblés": round(n_cibles), "dont partants": round(partants),
                   "% de partants captés": round(y_ord[:k].sum() / y_ord.sum() * 100, 1),
                   "coût (€)": round(cout_total), "clients sauvés": round(sauves, 1),
                   "profit net (€)": round(sauves * VALEUR - cout_total), "dans le budget": cout_total <= BUDGET})
plan = pd.DataFrame(lignes)
ok = plan[plan["dans le budget"]]
best = ok.loc[ok["profit net (€)"].idxmax()]
print(f"Budget {BUDGET:,} € · coût/client ciblé {COUT:.2f} € · valeur client {VALEUR} € · taux de succès {TAUX_SUCCES:.1%}")
print(f"➡ Meilleur ciblage dans le budget : top {best['% ciblé']} % — {best['clients ciblés']} clients, "
      f"{best['% de partants captés']} % des partants captés, profit net {best['profit net (€)']:,} €")
libre = plan.loc[plan["profit net (€)"].idxmax()]
print(f"   (sans contrainte de budget, l'optimum serait le top {libre['% ciblé']} % pour {libre['coût (€)']:,} € → profit {libre['profit net (€)']:,} €)")
fig, ax = plt.subplots()
ax.plot(plan["% ciblé"], plan["profit net (€)"], marker="o", color=COULEURS[0])
ax.axhline(0, color="grey", lw=1); ax.set_xlabel("% de la base ciblée (du plus risqué au moins risqué)")
ax.set_title("Profit net selon la taille du ciblage"); plt.show()
plan

## 9 · Synthèse pour le deck
Copiez ces chiffres dans votre présentation, puis rédigez **3 insights** au format : **constat + chiffre + donc…**

In [ ]:
print(f"MARQUE : {C['nom']}")
print(f"Base active : {len(data)} clients · départs à {C['fenetre']} j : {data.Y.mean():.1%}")
print(f"Variables utilisées : {len(VARIABLES)} · exclues : {EXCLURE}")
print(f"Modèle retenu : {MODELE_RETENU} · AUC test {res.loc[MODELE_RETENU, 'AUC test']} · AUC CV {res.loc[MODELE_RETENU, 'AUC validation croisée']}")
print(f"Next best offer : {nbo.to_dict()} · offre du plan : {OFFRE_RETENUE}")
print(f"Ciblage recommandé : top {best['% ciblé']} % · {best['clients ciblés']} clients · profit net estimé {best['profit net (€)']:,} €")

## 10 · Les défis experts
Le notebook vous a donné une première réponse. Les défis (voir votre cahier des charges) vous demandent d'aller **plus loin que les cellules pré-codées** : à vous d'écrire le code, seuls ou avec l'aide d'une IA, à condition de savoir l'expliquer.

In [ ]:
# 🧰 Boîte à outils pour les défis — objets déjà calculés plus haut :
#   data            : base active (une ligne par client, Y = 1 si départ)
#   X_tr, X_te, y_tr, y_te : découpage train / test
#   probas[nom]     : probabilités de départ des clients du test, par modèle
#   MODELES, VARIABLES, res : modèles entraînés, variables utilisées, tableau des AUC
#   uplift, gain_contact, cout_offre, nbo : résultats de la campagne test
#   act, mensuel, campagne, clients, digital : les fichiers chargés
#   C : paramètres du client (budget, cout_contact, valeur, fenetre)
BINS = {"velin": ([0, 34, 54, 200], ["18-34", "35-54", "55+"]), "atelier_nord": ([0, 29, 44, 200], ["18-29", "30-44", "45+"]),
        "spark": ([0, 25, 35, 200], ["18-25", "26-35", "36+"])}
def segment_client(df):
    if MARQUE in BINS:
        return pd.cut(df["age"], BINS[MARQUE][0], labels=BINS[MARQUE][1]).astype(str)
    return df["canal_principal" if MARQUE == "krousty" else "formule"]
seg_te = segment_client(data.loc[X_te.index])     # segment de chaque client du test
print(seg_te.value_counts())

In [ ]:
# ✋ À VOUS : défi 1

In [ ]:
# ✋ À VOUS : défi 2

In [ ]:
# ✋ À VOUS : défi 3

In [ ]:
# ✋ À VOUS : défi 4

In [ ]:
# ✋ À VOUS : défi 5 (propre à votre marque)